# First and Final Layer Attention Maps

Create two comparison figures: one for the first encoder layer and one for each model's final encoder layer. Each subpanel is a head-averaged token-by-token attention map with lead-boundary frames. The `dev_100hz` panel is intentionally left at its native 20-patch resolution.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / "src" / "encoder.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "encoder.py").is_file():
    ROOT = Path("/home/aimakeradmin/shady/TS-JEPA")
sys.path.insert(0, str(ROOT))

from collapse_investigation.analysis_utils import (
    LEADS_8,
    LEADS_12,
    attention_to_matrix,
    best_available_checkpoint,
    build_ours_encoder,
    lead_boundaries,
    load_encoder_weights,
    official_attention_hooks,
    prepare_signal,
)
from src.data.ptbxl_dataset import PTBXLDataset

OFFICIAL_ROOT = ROOT / "ecg_jepa"
if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))
from ecg_jepa import ecg_jepa

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PTBXL_DATA_DIR = ROOT / "data" / "ptbxl_500hz_2500_raw"
FIG_DIR = ROOT / "collapse_investigation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("device:", device)

In [ ]:
def build_official_encoder(ckpt_path: Path) -> torch.nn.Module:
    model = ecg_jepa(
        encoder_embed_dim=768,
        encoder_depth=12,
        encoder_num_heads=16,
        predictor_embed_dim=384,
        predictor_depth=6,
        predictor_num_heads=12,
        drop_path_rate=0.1,
        mask_scale=(0.175, 0.225),
        mask_type="block",
        pos_type="sincos",
        c=8,
        p=50,
        t=50,
    )
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.encoder.load_state_dict(ckpt["encoder"], strict=True)
    encoder = model.encoder.to(device).eval()
    for param in encoder.parameters():
        param.requires_grad_(False)
    print("Loaded official checkpoint epoch:", ckpt.get("epoch"))
    return encoder


def extract_official_attention(waves: torch.Tensor) -> list[torch.Tensor]:
    checkpoint = ROOT / "ecg_jepa" / "multiblock_epoch100.pth"
    encoder = build_official_encoder(checkpoint)
    with torch.no_grad(), official_attention_hooks(encoder) as captured:
        _ = encoder.representation(waves)
    return captured


def extract_ours_attention(run_name: str, waves: torch.Tensor) -> list[torch.Tensor]:
    run_dir = ROOT / "checkpoints" / run_name
    epoch, checkpoint, erank = best_available_checkpoint(run_dir, run_dir / "metrics.csv", "erank_ctx_pool")
    encoder, tokenizer, enc_cfg = build_ours_encoder(run_name, ROOT, use_flash=False)
    load_encoder_weights(encoder, checkpoint)
    encoder = encoder.to(device).eval()
    for param in encoder.parameters():
        param.requires_grad_(False)
    patches = tokenizer.patchify(waves)
    with torch.no_grad():
        _tokens, attn = encoder.forward_all(patches, return_attn=True)
    print(f"{run_name}: epoch {epoch}, pooled eRank {erank:.4f}, checkpoint {checkpoint.name}, layers {len(attn)}, patches {enc_cfg.num_patches}, leads {enc_cfg.num_leads}")
    return [a.detach().cpu() for a in attn]

In [ ]:
SAMPLE_INDEX = 0
ptbxl = PTBXLDataset(PTBXL_DATA_DIR, split="val", return_labels=True)
raw_wave, raw_label = ptbxl[SAMPLE_INDEX]
raw_batch = raw_wave.unsqueeze(0).to(device)
print("raw sample:", tuple(raw_batch.shape), "label:", raw_label.numpy())

waves = {
    "official": prepare_signal(raw_batch, "dev_preset_final"),
    "final_preset_final": prepare_signal(raw_batch, "final_preset_final"),
    "dev_preset_final": prepare_signal(raw_batch, "dev_preset_final"),
    "dev_100hz": prepare_signal(raw_batch, "dev_100hz"),
    "dev_12lead": prepare_signal(raw_batch, "dev_12lead"),
}
for name, wave in waves.items():
    print(name, tuple(wave.shape))

In [ ]:
attention = {
    "official": extract_official_attention(waves["official"]),
    "final_preset_final": extract_ours_attention("final_preset_final", waves["final_preset_final"]),
    "dev_preset_final": extract_ours_attention("dev_preset_final", waves["dev_preset_final"]),
    "dev_100hz": extract_ours_attention("dev_100hz", waves["dev_100hz"]),
    "dev_12lead": extract_ours_attention("dev_12lead", waves["dev_12lead"]),
}

for name, maps in attention.items():
    print(name, len(maps), tuple(maps[0].shape), tuple(maps[-1].shape))

In [ ]:
MODEL_PANELS = [
    {
        "key": "official",
        "title": "Official ECG-JEPA",
        "num_leads": 8,
        "num_patches": 50,
        "leads": LEADS_8,
    },
    {
        "key": "final_preset_final",
        "title": "Final preset",
        "num_leads": 8,
        "num_patches": 50,
        "leads": LEADS_8,
    },
    {
        "key": "dev_preset_final",
        "title": "Dev preset",
        "num_leads": 8,
        "num_patches": 50,
        "leads": LEADS_8,
    },
    {
        "key": "dev_100hz",
        "title": "Dev 100 Hz (20 patches)",
        "num_leads": 8,
        "num_patches": 20,
        "leads": LEADS_8,
    },
    {
        "key": "dev_12lead",
        "title": "Dev 12-lead",
        "num_leads": 12,
        "num_patches": 50,
        "leads": LEADS_12,
    },
]


def add_lead_frames(ax, num_leads: int, num_patches: int, leads: list[str]) -> None:
    boundaries, centers = lead_boundaries(num_leads, num_patches)
    for boundary in boundaries:
        ax.axhline(boundary - 0.5, color="white", linewidth=0.5, alpha=0.85)
        ax.axvline(boundary - 0.5, color="white", linewidth=0.5, alpha=0.85)
    ax.set_xticks(centers)
    ax.set_yticks(centers)
    ax.set_xticklabels(leads, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(leads, fontsize=7)
    ax.set_xlabel("key lead blocks", fontsize=8)
    ax.set_ylabel("query lead blocks", fontsize=8)


def plot_attention_row(layer_kind: str) -> None:
    if layer_kind not in {"first", "final"}:
        raise ValueError(layer_kind)
    fig, axes = plt.subplots(1, len(MODEL_PANELS), figsize=(18, 4.2), constrained_layout=True)
    for ax, panel in zip(axes, MODEL_PANELS):
        maps = attention[panel["key"]]
        layer_idx = 0 if layer_kind == "first" else len(maps) - 1
        matrix = attention_to_matrix(maps[layer_idx], sample=0)
        im = ax.imshow(matrix, cmap="viridis", aspect="equal", interpolation="nearest")
        ax.set_title(f"{panel['title']}\nlayer {layer_idx}", fontsize=10)
        add_lead_frames(ax, panel["num_leads"], panel["num_patches"], panel["leads"])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    fig.suptitle(f"Head-Averaged Patch Attention: {layer_kind.title()} Encoder Layer", fontsize=13)
    out = FIG_DIR / f"attention_{layer_kind}_layer_comparison.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print("saved", out)
    plt.show()

plot_attention_row("first")
plot_attention_row("final")